In [4]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent
load_dotenv()

True

In [5]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [12]:
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

agent_graph = create_agent(
    model = gemini_model,
    tools = [get_product],
    system_prompt = "You are helpful product assistant at a tech store"
)

def ask(question: str):
    result = agent_graph.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [13]:
ask("what is the price of wireless headphones.")

The wireless headphones are $79.99.


In [14]:
ask("tell me about mechanical keyboard")

A mechanical keyboard typically costs $129. It has a rating of 4.8. This particular model is tenkeyless, uses Cherry MX Brown switches, and features per-key RGB lighting.


In [15]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [17]:
agent2 = create_agent(
    gemini_model,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [18]:
ask2("how do people like smart watch")

[{'type': 'text', 'text': 'The smart watch has 340 reviews and an average rating of 3.9.', 'extras': {'signature': 'CqYDAQw51sdw8zMpVezHskqplqf67uQFsTIHAjds3jS5OYFrSMuHmb2iojTJcLxfovsabh/HVjLphyTBVusa6ye0YhAC22f+v+I7rum7Mf5vkhQjA1EH6Kdz9oDBv60HTDiT/rYatjdXz/NCz1EUNo0mOZv2F4U9jdMagYQc69/QiIgMGDADmKLFSNUam3qnjgDPjnKZwzphERVrQGR+c030REeSwWxi+mEJe3W9jDbDFABJpOxtV3AGU7/c7BdPTxpvdSuuU4SfbZ25H3IvOcmY0NC3hWx3Vtx5ZpzNnSeO8H/4Wmk3nG6aACbf5+nIje4Qda8/QvBUNx81rpN/cKv03WZYlbpAA9yh3jxwuEzc02XrG07dnbaVzKHxdU3onqJKJcO1tMsIZ4Vql6RPg6p7ORvHWOTLUUupeBEsmnYt5+GzUTYOr1X0y4UZT/Btuc7MnoiJC/l5aSYKSguMIK5lZ6l+O0A+kZmn/DMUcjpmMpvB1YDik61LdJO79k2aIAhaa4e3TNOMsexVg5mvlPV/xDSkDoMAYT4X3QVG6ikFP2/yAsOIhS4='}}]


In [19]:
ask2("what is the price and reviews of smart watch")

The smart watch is priced at $199.99. It has 340 reviews with an average rating of 3.9.


In [20]:
ask2("what are the reviews on this product")

[{'type': 'text', 'text': 'Which product are you interested in?', 'extras': {'signature': 'CrIEAQw51seDJvWo+W8/Dsz7hKli6VI2C50An1XL+bAwO2ssy9/o46B+1BvtIC5GcxMzZG6l8LLtffL/GVk7QwekEgr2jQ9FUTjnRlf6a9LUce6Ntx+IbvZlLSXMXKxQBOAMwR9y3I+PA23LZoC0Gd1MehICmEiMkt9i6nF6KPzKnboKKxC4vRKNGWXbhvaZHRyR4VOdQITQK98wLew19UJ6OSG1an0COR5gxuVRnd6Hrddvyj+lhxD+s+PxMue0OYWjrIqT/n09ZOo+kzyDsOonkJRsbJuf/rnRlEVP5POw8n+ySCXJyDa8oLyxqKFMD5PAiu2YtQ1sKDYwHgn44pKWkwmTH0GYV/PQ6gcNe37GtgNjjXG+RUc5DkgeNbEuHMuzwEZzcSoe8a9o5vWBM/2e0G6U248sYQhdk+CcmgnpahyD+mK3YawIudEKX2o2ZEFH6QasOXnR8ssuj6EUTGAkje6v3kqtRkXrkEvHGV2uWOpQZD7bPXmGEbz7FL+HjnueV/T01zk8KPH1oJx2MZ81r/wiTYGUIpomHktEHqkS2YU96fuLiY+1M1GPWaUuFHWLUAX0cRobRriCWwBxSSWPQnrFwTni3H49GKWWL/uQEonLvAyXAVfYcMflPaNaKmmI7aa0kKct8tbT0y6/sWlg+3bgh/QmyMcrXrrrWOXAKejncgMT+Vkk+I5A0jLtkou1Hxr1TX4wAlniMZOBYoSaDNEaer5fn2Oi3KI1/UgUN8pHAQ=='}}]
